<a href="https://colab.research.google.com/github/HaaseSchuetz/llm-quantization-benchmark/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Quantization Benchmark Demo

**Benchmark quantization methods for LLMs in Colab!**

 **GPU Required** (T4 or A100 recommended).
 **Estimated Time:** ~20–40 mins (depending on methods).
 **Storage:** ~20GB (for Mistral-7B).

---## What You’ll Do
1. Install dependencies.
2. Benchmark **Mistral-7B** with **FP16, INT8, and INT4**.
3. Compare **memory, latency, and accuracy**.
4. Generate a **report with visualizations**.

---## Setup
Click **Runtime → Run all** to execute the entire notebook.

## 1 Install Dependencies

In [ ]:
# Install required packages
!pip install -q \
    torch==2.1.2 \
    transformers==4.38.2 \
    accelerate==0.25.0 \
    bitsandbytes==0.41.3 \
    datasets==2.16.1 \
    lm-evaluation-harness==0.4.0 \
    pandas \
    matplotlib \
    seaborn \
    tqdm \
    psutil \
    GPUtil

# Restart kernel (Colab-specific)
import os
if 'COLAB_GPU' in os.environ:
    print(' Restarting kernel...')
    os.kill(os.getpid(), 9)

## 2 Imports and Setup

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from quantization.methods import get_quantizer
from quantization.metrics.memory import MemoryBenchmark
from quantization.metrics.latency import LatencyBenchmark
from evaluation.evaluator import LLEvaluator
import torch

# Clone the repo
!git clone https://github.com/HaaseSchuetz/llm-quantization-benchmark.git 2>/dev/null || echo "Repo already cloned."
%cd llm-quantization-benchmark

# Add to path
sys.path.append(str(Path.cwd()))

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f' Using device: **{device}**')
if device == "cuda":
    print(f' GPU: {torch.cuda.get_device_name(0)}')
    print(f' GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')

## 3 Benchmark Mistral-7B with Different Quantization Methods

In [ ]:
from scripts.benchmark import benchmark_quantization

# Define prompts for latency/throughput tests
prompts = [
    "Explain the concept of quantization in machine learning.",
    "Write a Python function to reverse a string.",
    "What are the benefits of using LLMs in production?",
    "Summarize the key ideas of parameter-efficient fine-tuning."
]

# Run benchmark (limit to 50 examples for speed)
results = benchmark_quantization(
    model_name="mistral-7b",
    quantization_methods=["fp16", "int8", "int4"],
    benchmarks=["mmlu"],
    save_dir="colab_results",
    limit=50,
    prompts=prompts
)

## 4 Analyze Results

In [ ]:
# Load results
with open("colab_results/raw/mistral-7b_quantization_benchmark.json") as f:
    results = json.load(f)

# Extract efficiency metrics
efficiency_data = []
for result in results:
    q_name = result["config"]["name"]
    q_bits = result["config"]["bits"]
    eff = result["efficiency"]
    efficiency_data.append({
        "Quantization": q_name,
        "Bits": q_bits,
        "Model Size (MB)": eff["memory"]["model_size_mb"],
        "GPU Memory Used (MB)": eff["memory"]["gpu_used_mb"],
        "Avg Latency (ms)": eff["latency"]["avg_latency_ms"],
        "Tokens/sec": eff["token_speed"]
    })

# Extract accuracy metrics
accuracy_data = []
for result in results:
    q_name = result["config"]["name"]
    q_bits = result["config"]["bits"]
    for acc_result in result["accuracy"]:
        accuracy_data.append({
            "Quantization": q_name,
            "Bits": q_bits,
            "Benchmark": acc_result["benchmark"],
            "Accuracy": acc_result["metrics"]["accuracy"]
        })

# Create DataFrames
eff_df = pd.DataFrame(efficiency_data)
acc_df = pd.DataFrame(accuracy_data)

# Merge for combined analysis
merged_df = pd.merge(eff_df, acc_df, on=["Quantization", "Bits"])

# Display results
print("### Efficiency Metrics")
print(eff_df.to_markdown(index=False))
print("\n### Accuracy Metrics")
print(acc_df.to_markdown(index=False))

## 5 Visualize Trade-offs

In [ ]:
# Plot 1: Memory vs. Accuracy
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=merged_df,
    x="Model Size (MB)",
    y="Accuracy",
    hue="Bits",
    size="Bits",
    sizes=(200, 400, 600),
    style="Quantization",
    alpha=0.8
)
plt.title("Memory vs. Accuracy Trade-off\n(Smaller = Better Memory, Higher = Better Accuracy)")
plt.xlabel("Model Size (MB)")
plt.ylabel("Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 2: Latency vs. Accuracy
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=merged_df,
    x="Avg Latency (ms)",
    y="Accuracy",
    hue="Bits",
    size="Bits",
    sizes=(200, 400, 600),
    style="Quantization",
    alpha=0.8
)
plt.title("Latency vs. Accuracy Trade-off\n(Lower = Better Latency, Higher = Better Accuracy)")
plt.xlabel("Avg Latency (ms)")
plt.ylabel("Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot 3: Throughput vs. Accuracy
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=merged_df,
    x="Tokens/sec",
    y="Accuracy",
    hue="Bits",
    size="Bits",
    sizes=(200, 400, 600),
    style="Quantization",
    alpha=0.8
)
plt.title("Throughput vs. Accuracy Trade-off\n(Higher = Better Throughput, Higher = Better Accuracy)")
plt.xlabel("Tokens/sec")
plt.ylabel("Accuracy")
plt.grid(True)
plt.tight_layout()
plt.show()

## 6 Key Insights
### Observations from Mistral-7B:
1. **Memory Savings**:
   - **INT8**: ~50% reduction vs. FP16 (14GB → 7GB).
   - **INT4**: ~75% reduction vs. FP16 (14GB → 3.5GB).

2. **Speed**:
   - **INT4**: ~1.5–2x faster than FP16.
   - **INT8**: ~1.2–1.5x faster than FP16.

3. **Accuracy Trade-off**:
   - **INT8**: Minimal drop (~1–2%).
   - **INT4**: ~3–5% drop (varies by benchmark).

4. **Best for Production**:
   - **INT8**: Best balance for cloud deployment.
   - **INT4**: Best for edge devices (if accuracy drop is acceptable).
   - **GPTQ/AWQ**: Better accuracy than standard INT4.

---## Next Steps
### To Run Locally:
1. Clone the repo: `git clone https://github.com/HaaseSchuetz/llm-quantization-benchmark.git`
2. Install dependencies: `pip install -r requirements.txt`
3. Benchmark: `python scripts/benchmark.py --model mistral-7b --quantizations int4 int8 fp16`

### To Extend:
- **Add GPTQ/AWQ**: Install `auto-gptq` and `auto-awq`, then add to `config/quantization.json`.
- **Add more benchmarks**: Edit `config/benchmarks.json`.
- **Compare more models**: Add to `config/models.json`.

---** Star the [repo](https://github.com/HaaseSchuetz/llm-quantization-benchmark) if you found this useful!**